# Semana 11 · Sesión 1: Del módulo al paquete, y sus pruebas

**Módulo 3**

## Objetivos de la sesión

1. Organizar un módulo que creció en un **paquete** de Python, con
   `__init__.py`.
2. Escribir **pruebas de pytest** para funciones simbólicas, comparando por
   equivalencia simbólica y no por cómo se ve la expresión.
3. **Documentar** funciones con docstrings que otra persona pueda leer con
   `help()`.

## Retomamos

En la semana 10 sacamos el oscilador amortiguado de un notebook largo y lo
pusimos en `oscilador.py`. Le agregamos la energía y ya empezaba a crecer: un
proyecto de verdad acaba con decenas de funciones. Hoy damos dos pasos más:

- Partir el módulo en un **paquete**, una carpeta con varios archivos `.py`.
- Escribir **pruebas** que comprueben el paquete solas, cada vez que algo
  cambie, sin que nadie tenga que revisar celdas a ojo.

Desde la semana 3 tus tareas pasan por pruebas de pytest (el check ✅ o ❌ de
tu PR). Hoy tú escribes las pruebas.

In [ ]:
import subprocess
import sys

import pytest
import sympy as sp

sp.init_printing()

## De un módulo a un paquete

Un **paquete** es una carpeta de módulos con un archivo especial,
`__init__.py`. Junto a este notebook está la carpeta
[`osciladores/`](osciladores/): es el `oscilador.py` de la semana 10 partido
en dos archivos, según de qué trata cada función.

![Estructura del paquete osciladores: __init__.py, amortiguado.py y energia.py, junto a la carpeta tests y el notebook](img/estructura-paquete.svg)

Ábrelos en JupyterLab. `amortiguado.py` tiene lo de la semana 10;
`energia.py`, la energía mecánica. Y `__init__.py` casi no tiene código: un
docstring y una línea `from ... import ...`.

Como en la semana 10, activamos `autoreload` antes de importar, para que los
cambios a los `.py` se vean sin reiniciar el kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import osciladores

`import osciladores` **ejecuta `__init__.py`**, y lo que ese archivo define o
importa es lo que queda disponible con el prefijo `osciladores.`:

In [ ]:
print([nombre for nombre in dir(osciladores) if not nombre.startswith("_")])

Están las tres funciones de `amortiguado.py`, los símbolos `t` y `x`, y el
propio submódulo `amortiguado`. Falta `energia`: `__init__.py` no lo importa,
así que hay que pedirlo aparte.

In [ ]:
from osciladores import energia

print([nombre for nombre in dir(energia) if not nombre.startswith("_")])

¿Para qué sirve `__init__.py` entonces? Para decidir qué ve **quien usa** el
paquete. Con la línea `from osciladores.amortiguado import ...`, nadie necesita
saber en qué archivo vive `resolver`: escribe `osciladores.resolver` y ya.
Mañana puedes mover `resolver` a otro archivo y, si `__init__.py` lo sigue
importando, ningún notebook se rompe.

El docstring de `__init__.py` es la documentación del paquete entero:

In [ ]:
print(osciladores.__doc__)

## TODO en clase 1

Haz que `osciladores.energia_mecanica` funcione sin importar `energia` aparte.

1. En `osciladores/__init__.py`, agrega una línea que importe
   `energia_mecanica` desde `osciladores.energia`. Guarda el archivo.
2. Agrega al docstring del paquete que `energia_mecanica` también se obtiene
   directo de `osciladores`.
3. Corre la celda de abajo. Antes de tu cambio imprime un `AttributeError`;
   después, el docstring de la función.

*Pista:* copia la forma de la línea que ya está. Si `autoreload` no se entera
del cambio, **Kernel → Restart Kernel** y corre el notebook desde arriba.

In [ ]:
# TODO en clase: después de editar __init__.py, esto ya no da error
try:
    print(osciladores.energia_mecanica.__doc__)
except AttributeError as error:
    print("AttributeError:", error)

## De un `assert` a una prueba

Ya sabes comprobar un resultado en una celda. Sin fricción, con $m = 1$ y
$k = 4$, la masa oscila con $\omega = 2$; si parte del reposo en $x = 1$, su
posición es $\cos 2t$:

In [ ]:
t = osciladores.t
posicion = osciladores.resolver(1, 0, 4, 1, 0)   # m, b, k, x0, v0

display(sp.Eq(osciladores.x(t), posicion))
assert posicion == sp.cos(2 * t)

El `assert` no dijo nada: eso significa que se cumplió. Si no, habría lanzado
un `AssertionError`.

Pero cuidado con `==` entre expresiones de SymPy. Compara **cómo están
escritas** (sus árboles), no si valen lo mismo:

In [ ]:
identidad = sp.sin(t)**2 + sp.cos(t)**2

print(identidad == 1)                    # dos árboles distintos
print(sp.simplify(identidad - 1) == 0)   # la diferencia se simplifica a cero
print(identidad.equals(1))               # SymPy prueba si valen lo mismo

Por eso todas las pruebas del curso restan y simplifican:
`sp.simplify(resultado - esperado) == 0`. Así una respuesta correcta escrita de
otra forma no se da por mala. El `assert posicion == sp.cos(2 * t)` de arriba
funcionó de suerte, porque `resolver` simplificó justo a esa forma.

Un `assert` en una celda tiene otro problema: solo corre si alguien corre la
celda. Una **prueba** es una función que pytest encuentra y corre sola:

- Vive en un archivo cuyo nombre empieza con `test_`, dentro de `tests/`.
- Es una función cuyo nombre también empieza con `test_` y **dice qué
  comprueba**: `test_resolver_sin_friccion_da_un_coseno`, no `test_1`.
- Prepara los datos, llama al paquete y termina con `assert`. Esas tres partes
  van separadas por una línea en blanco.

Abre [`tests/test_osciladores.py`](tests/test_osciladores.py). Esta es la
primera prueba que trae:

```python
def test_resolver_sin_friccion_da_un_coseno():
    # m = 1, k = 4: omega = 2. Parte del reposo en x = 1.
    posicion = osciladores.resolver(1, 0, 4, 1, 0)

    assert sp.simplify(posicion - sp.cos(2 * t)) == 0
```

## Correr pytest

En una terminal, desde esta carpeta (`semana-11/clase/`):

```bash
python -m pytest tests -v
```

pytest busca en `tests/` los archivos y las funciones `test_...`, las corre y
reporta cada una. `-v` (*verbose*) pide una línea por prueba. Usamos
`python -m pytest` y no solo `pytest` porque así Python agrega la carpeta
actual a la ruta donde busca paquetes, y `import osciladores` funciona.

Para no salir del notebook, esta función corre el mismo comando en un proceso
aparte con `subprocess` e imprime lo que habría salido en la terminal. Es
Python normal, no una celda mágica: en un proceso aparte, pytest lee los `.py`
recién guardados y no lo que este kernel ya tiene en memoria.

In [ ]:
def correr_pruebas():
    # Un proceso aparte: lo mismo que python -m pytest tests -v --tb=short
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "tests", "-v", "--tb=short"],
        capture_output=True,
        text=True,
    )
    print(resultado.stdout)

In [ ]:
correr_pruebas()

## Leer un fallo

Cada línea dice el nombre de la prueba y su resultado: `PASSED`, `FAILED` o
`SKIPPED` (las que quedan pendientes para hoy). Dos fallaron, y las dos son de
la energía. Lee el reporte de `test_energia_sin_friccion_se_conserva` de abajo
hacia arriba:

- La última línea, `assert -4*sin(4*t) == 0`, dice qué se esperaba y qué hubo:
  la derivada de la energía debía ser cero y no lo es.
- Arriba, la línea del archivo de pruebas que falló.

Sin fricción la energía **tiene** que conservarse. Si no se conserva, el error
no está en la física sino en `energia.py`. Compara su docstring con lo que de
verdad calcula la última línea: `energia.py` trae un error **a propósito**, el
tipo de error que una prueba atrapa y una revisión a ojo se salta.

La otra prueba que falla lo confirma por otro lado: en reposo en $x = 1$, toda
la energía está en el resorte, $\tfrac12 k x_0^2 = 2$, y la función da $4$.

## TODO en clase 2

1. Corrige el error de `osciladores/energia.py` y guarda.
2. En `tests/test_osciladores.py`, escribe
   `test_resolver_cumple_la_ecuacion_y_las_condiciones_iniciales` (borra la
   línea `pytest.skip`). Para el caso subamortiguado $m = 1$, $b = 1$,
   $k = 4$, $x_0 = 1$, $v_0 = 0$:
   - Sustituye la `posicion` de `osciladores.resolver` en el lado izquierdo de
     `osciladores.ecuacion_de_movimiento(...)` y comprueba que se simplifica a
     $0$.
   - Comprueba que $x(0) = x_0$ y que $\dot x(0) = v_0$.
3. Corre la celda de abajo: deben pasar las cinco primeras pruebas.

*Pista:* `ecuacion.lhs.subs(x(t), posicion).doit()` sustituye y calcula las
derivadas. Una prueba puede tener varios `assert`; la primera que falle es la
que se reporta.

In [ ]:
# TODO en clase: corrige energia.py, escribe la prueba y vuelve a correr
correr_pruebas()

## Un caso que debe fallar: `pytest.raises`

Desde la semana 10, `regimen` lanza un `ValueError` si le das símbolos, porque
con símbolos no se sabe el signo de $b^2 - 4mk$. Ese aviso también es parte de
lo que promete la función, y también se prueba. `pytest.raises` es un bloque
`with` que **exige** que adentro se lance el error:

In [ ]:
m, b, k = sp.symbols("m b k", positive=True)   # constantes físicas: positivas

with pytest.raises(ValueError) as error:
    osciladores.regimen(m, b, k)

print("Se lanzó ValueError:", error.value)

Si `regimen` regresara un régimen en lugar de avisar, el bloque mismo
fallaría con `Failed: DID NOT RAISE`.

## Una prueba, varios casos: `parametrize`

Para probar los tres regímenes, escribir tres pruebas casi iguales sería
copiar y pegar otra vez. `@pytest.mark.parametrize` corre **la misma** prueba
con varios datos, y pytest reporta cada caso por separado. Por ejemplo, para la
posición inicial:

```python
@pytest.mark.parametrize("x0", [1, 2, sp.Rational(1, 2)])
def test_resolver_parte_de_x0(x0):
    posicion = osciladores.resolver(1, 1, 4, x0, 0)

    assert sp.simplify(posicion.subs(t, 0) - x0) == 0
```

El primer argumento nombra los parámetros, separados por coma si son varios
(`"friccion, esperado"`); el segundo es la lista de casos, una tupla por caso
cuando hay más de un parámetro. Este decorador solo funciona cuando pytest corre
la prueba: dentro de una celda, la función se llamaría sin sus datos.

## TODO en clase 3

En `tests/test_osciladores.py` (borra cada `pytest.skip`):

1. `test_regimen_de_cada_friccion`, con `parametrize`: con $m = 1$ y $k = 4$,
   la fricción $b = 1$ da `"subamortiguado"`, $b = 4$ da `"crítico"` y $b = 5$
   da `"sobreamortiguado"`.
2. `test_regimen_con_simbolos_avisa`, con `pytest.raises`.

Corre la celda de abajo: deben pasar **todas**, y la paramétrica aparece tres
veces en el reporte, una por caso.

*Pista:* aquí se compara una cadena con otra cadena, no expresiones de SymPy;
ahí `==` sí es lo correcto.

In [ ]:
# TODO en clase: todas las pruebas pasan
correr_pruebas()

## Docstrings

Las pruebas dicen **si** el paquete funciona; los docstrings dicen **qué hace**
cada función, para quien la va a usar sin leer su código. Compara los dos de
`osciladores`:

In [ ]:
help(osciladores.ecuacion_de_movimiento)
help(energia.energia_mecanica)

El primero basta para una función corta. El segundo es el que piden las
funciones de tu proyecto, y sigue una forma común en Python científico:

- **Una primera línea** que dice qué calcula. Es la que aparece en las listas.
- **Parámetros**: cada argumento, qué representa físicamente y qué tipo de
  objeto espera (un número, un símbolo, una expresión en `t`).
- **Devuelve**: qué regresa y qué propiedades tiene. Aquí, que sin fricción
  la energía es constante: eso es justo lo que comprueba una prueba.
- La **motivación física**, cuando no es obvia.

Docstrings así no son solo para `help()`. Herramientas como **Sphinx** o
**MkDocs** (con su extensión *mkdocstrings*) los leen de todo un paquete y
generan un sitio web de documentación, como el de
[SymPy](https://docs.sympy.org/). No las vamos a instalar en el curso; lo que
importa es que la documentación vive **junto al código**, en el docstring, y no
en un archivo aparte que nadie actualiza.

## Resumen

Un **paquete** es una carpeta de módulos con `__init__.py`; `import paquete`
ejecuta ese archivo, y lo que importa es lo que ve quien usa el paquete.

Una **prueba** es una función `test_...` en `tests/test_*.py` que termina con
`assert`, y `python -m pytest tests -v` las corre todas. Con SymPy se compara
por equivalencia, `sp.simplify(resultado - esperado) == 0`, nunca con `==`
entre expresiones. `pytest.raises` prueba los errores que la función promete
lanzar y `@pytest.mark.parametrize` corre una prueba con varios casos. Un
reporte de fallo se lee de abajo hacia arriba.

Un **docstring** completo dice qué calcula la función, sus parámetros, qué
devuelve y por qué, y lo leen `help()`, Sphinx y MkDocs.

**Siguiente sesión:** que las pruebas corran solas en GitHub cada vez que
haces `push` (integración continua), cómo organizar lo pendiente con issues y
milestones, y la revisión del PR preliminar de tu proyecto intermedio.